# Simple linear regression from scratch

The closed-form (OLS) formulas for m and b, written as our own class and compared with scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("data/placement.csv")
X = df["cgpa"].to_numpy()        # 1-D arrays are enough for our own class
y = df["package"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

## The pieces of the formula, on the training set

In [ ]:
x_bar, y_bar = X_train.mean(), y_train.mean()
num = ((X_train - x_bar) * (y_train - y_bar)).sum()   # top of the m formula
den = ((X_train - x_bar) ** 2).sum()                   # bottom of the m formula
m = num / den
b = y_bar - m * x_bar
print(f"x_bar={x_bar:.4f} y_bar={y_bar:.4f} num={num:.3f} den={den:.3f}")
print(f"m={m:.4f} b={b:.4f}")

## Our own class

In [ ]:
class MyLR:
    def fit(self, X, y):
        # closed-form OLS: no loops needed with NumPy
        x_bar, y_bar = X.mean(), y.mean()
        self.m = ((X - x_bar) * (y - y_bar)).sum() / ((X - x_bar) ** 2).sum()
        self.b = y_bar - self.m * x_bar
        return self

    def predict(self, X):
        return self.m * X + self.b

mine = MyLR().fit(X_train, y_train)
mine.m, mine.b

## Same answer as scikit-learn

In [ ]:
sk = LinearRegression().fit(X_train.reshape(-1, 1), y_train)
print(sk.coef_[0], sk.intercept_)
print(mine.predict(X_test[:3]).round(4))
print(sk.predict(X_test[:3].reshape(-1, 1)).round(4))

## Checks: at the best line, the two derivatives are zero

In [ ]:
d = y_train - mine.predict(X_train)      # errors
print("sum of errors          :", round(d.sum(), 10))            # dE/db = -2 * sum(d) = 0
print("sum of errors times x  :", round((X_train * d).sum(), 10)) # dE/dm = -2 * sum(x d) = 0
print("E at the best line     :", round((d ** 2).sum(), 2))
print("E with m + 0.01        :", round(((y_train - ((mine.m + 0.01) * X_train + mine.b)) ** 2).sum(), 2))